# MFPT Sweep Analysis

Interactive analysis of Markov sweep outputs.  Loads every (dataset × parameter combination) under a `storage_root` into one tidy DataFrame and lets you compare MFPT distributions along any sweep axis (protamine, Eads, TNP2, ...) and across datasets.

**Workflow**
1. Point at a sweep `storage_root` and load.
2. Inspect which parameters varied.
3. Pick a slice (fix all axes except the ones you want to compare).
4. Plot — distributions, sweep curves, heatmaps, dataset comparisons.

All loading + filtering helpers live in [src/analysis/sweep_loader.py](../src/analysis/sweep_loader.py).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

from src.analysis.sweep_loader import (
    load_sweep, attach_sequence_features,
    varying_columns, cross_dataset_axes, dataset_determined_columns,
    filter_df, add_dataset_class,
)

PROJECT_ROOT = Path('/home/pol_schiessel/maya620d/pol/Projects/Codebase/Spermatogensis')

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('imports OK')

## 1. Load a sweep

Set `STORAGE_ROOT` to whatever sweep directory you want to analyse (the same value you put under `storage_root:` in the sweep YAML).

In [ ]:
STORAGE_ROOT = PROJECT_ROOT / 'output/markov_output/eads_weighted_sweep'

# Keep rows with mfpt = inf so right-censored/non-falling nucleosomes can be
# counted and shown in the plots. Set False to recover the old finite-only load.
LOAD_CENSORED_MFPT = True

df = load_sweep(
    STORAGE_ROOT,
    drop_nonfinite=not LOAD_CENSORED_MFPT,
    drop_nonpositive_mfpt=not LOAD_CENSORED_MFPT,
)
if not df.empty:
    df['mfpt_censored'] = ~np.isfinite(df['mfpt']) | (df['mfpt'] <= 0)
    df.loc[df['mfpt_censored'], 'log10_mfpt'] = np.nan

print(f'Loaded {len(df):,} MFPT rows from {STORAGE_ROOT}')
print(f'  datasets: {sorted(df["dataset"].unique()) if not df.empty else "(empty)"}')
print(f'  varying axes: {varying_columns(df)}')
if 'mfpt_censored' in df.columns:
    print(f'  censored/non-falling rows: {int(df["mfpt_censored"].sum()):,}')
df.head()


## 2. Optional: attach sequence features (GC, N_CpG, ρ_CpG)

Joining the per-dataset FASTA lets you slice MFPT distributions by composition (e.g. CpG bins for the v=0/v=1 swap test).  Skip this cell if you only want pure MFPT plots.

In [ ]:
HAMNUCRET_FASTA_DIR = Path('/home/pol_schiessel/maya620d/pol/Projects/Codebase/HAMNucRetSeq_pipeline')

DATASET_FASTA = {
    # Stable-147 refined datasets from HAMNucRetSeq_pipeline
    'ret_all_stable147_refined'                    : {'path': HAMNUCRET_FASTA_DIR / 'RET_output/all/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa', 'id_style': 'name'},
    'ctrl01_random_genome_safe_stable147_refined'  : {'path': HAMNUCRET_FASTA_DIR / 'SOM_output/control_sets/ctrl01_random_genome_safe/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa', 'id_style': 'name'},
    'ctrl02_random_genome_gcmatched_stable147_refined': {'path': HAMNUCRET_FASTA_DIR / 'SOM_output/control_sets/ctrl02_random_genome_gcmatched/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa', 'id_style': 'name'},
    'ctrl03_som_gcmatched_stable147_refined'       : {'path': HAMNUCRET_FASTA_DIR / 'SOM_output/control_sets/ctrl03_som_gcmatched/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa', 'id_style': 'name'},
    'ctrl04_bound_prom_evicted_stable147_refined'  : {'path': HAMNUCRET_FASTA_DIR / 'SOM_output/control_sets/ctrl04_bound_prom_evicted/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa', 'id_style': 'name'},
    'ctrl05_unbound_prom_yazdi_stable147_refined'  : {'path': HAMNUCRET_FASTA_DIR / 'SOM_output/control_sets/ctrl05_unbound_prom_yazdi/w167/fully_wrapped/crystal_freedna_md/stablepos/stable147.fa', 'id_style': 'name'},

    # # Legacy datasets retained for old sweep outputs
    # 'ret_single_nuc'                  : {'path': HAMNUCRET_FASTA_DIR / 'RET_data/single_nuc/stable_147/stable147.fa', 'id_style': 'name'},
    # 'ret_intermediate_nuc'            : {'path': 'hamnucret_data/SPRM_IN_SEQ/RET_data/intermediate/pooled_peaks_intermediate_nuc147.fa', 'id_style': 'name'},
    # 'ret_broad_nuc'                   : {'path': 'hamnucret_data/SPRM_IN_SEQ/RET_data/broad_ambiguous/pooled_peaks_broad_ambiguous_nuc147.fa', 'id_style': 'name'},
    # 'ctrl01_random_genome_safe'       : {'path': 'hamnucret_data/SPRM_IN_SEQ/SOM_data/control_sets/ctrl01_random_genome_safe/ctrl01.fa',     'id_style': 'coord'},
    # 'ctrl02_random_genome_gcmatched'  : {'path': 'hamnucret_data/SPRM_IN_SEQ/SOM_data/control_sets/ctrl02_random_genome_gcmatched/ctrl02.fa', 'id_style': 'coord'},
    # 'ctrl03_som_gcmatched'            : {'path': 'hamnucret_data/SPRM_IN_SEQ/SOM_data/control_sets/ctrl03_som_gcmatched/ctrl03.fa',           'id_style': 'name'},
    # 'ctrl04_bound_prom_evicted'       : {'path': 'hamnucret_data/SPRM_IN_SEQ/SOM_data/control_sets/ctrl04_bound_prom_evicted/ctrl04.fa',       'id_style': 'name'},
    # 'ctrl05_unbound_prom_random'      : {'path': 'hamnucret_data/SPRM_IN_SEQ/SOM_data/control_sets/ctrl05_unbound_prom_yazdi/ctrl05.fa',       'id_style': 'name'},
}

df = attach_sequence_features(df, DATASET_FASTA, project_root=PROJECT_ROOT)
print(f'  with sequences: {df["n_cpg"].notna().sum():,} of {len(df):,} rows')
df.head()


In [ ]:
df["dataset"].unique()

## 3. RET vs control comparison at one parameter combination

Pool RET into one distribution and overlay it on each control as small multiples (KDE on top, ECDF below).  Each panel reports the two-sample KS test and Δmedian(log10 MFPT).

> **Slicing gotcha.** `eads_apply`, `eads_delta`, `eads_weight_mode` are *dataset-class-determined* by the sweep's `_normalized_combo_for_dataset` rule (RET → `False/0/none`, controls → `True/<delta>/<mode>`).  Filtering on them in a cross-dataset slice silently drops one class.  Stick to **real** sweep axes — `cross_dataset_axes(df)` lists them.  The cell below also prints per-dataset row counts after filtering so you catch any imbalance immediately.

In [ ]:
GROUP_COLORS = {
    # 'ctrl01_random_genome_safe'      : '#aec6cf',
    'ctrl01_random_genome_safe_stable147_refined'      : '#aec6cf',
    # 'ctrl02_random_genome_gcmatched' : '#77b5fe',
    'ctrl02_random_genome_gcmatched_stable147_refined' : '#77b5fe',
    # 'ctrl03_som_gcmatched'           : '#4169e1',
    'ctrl03_som_gcmatched_stable147_refined'           : '#4169e1',
    # 'ctrl04_bound_prom_evicted'      : '#0047ab',
    'ctrl04_bound_prom_evicted_stable147_refined'      : '#0047ab',
    # 'ctrl05_unbound_prom_random'     : '#003366',
    'ctrl05_unbound_prom_yazdi_stable147_refined'      : '#003366',
    # 'ret_single_nuc'                 : '#ff6961',
    'ret_all_stable147_refined'       : '#d35400',
    # 'ret_intermediate_nuc'           : '#c0392b',
    # 'ret_broad_nuc'                  : '#7b241c',
}
RET_COLOR  = '#c0392b'
CTRL_COLOR = '#0047ab'

RET_DATASETS  = ['ret_all_stable147_refined']
CTRL_DATASETS = ['ctrl01_random_genome_safe_stable147_refined',
                 'ctrl02_random_genome_gcmatched_stable147_refined',
                 'ctrl03_som_gcmatched_stable147_refined',
                 'ctrl04_bound_prom_evicted_stable147_refined',
                 'ctrl05_unbound_prom_yazdi_stable147_refined']

# Plot option for mfpt = inf rows. These are right-censored/non-falling
# nucleosomes. They cannot enter KDE/median/KS directly, but they can be counted
# and shown at the right edge of each panel.
SHOW_CENSORED_MFPT = True
CENSORED_X_PAD = 0.08


def _mfpt_arrays(frame, col='log10_mfpt', show_censored=SHOW_CENSORED_MFPT):
    """Return finite log10(MFPT), censored count, and ECDF denominator."""
    if frame.empty or col not in frame.columns:
        return np.array([]), 0, 0
    vals = pd.to_numeric(frame[col], errors='coerce').to_numpy(dtype=float)
    finite = np.isfinite(vals)
    finite_vals = vals[finite]
    if show_censored:
        if 'mfpt_censored' in frame.columns:
            censored_n = int(frame['mfpt_censored'].fillna(False).sum())
        else:
            censored_n = int((~finite).sum())
    else:
        censored_n = 0
    return finite_vals, censored_n, int(finite_vals.size + censored_n)


def _censored_label(finite_n, censored_n, show_censored=SHOW_CENSORED_MFPT):
    if show_censored and censored_n:
        return f'n={finite_n + censored_n} ({finite_n} finite, {censored_n} censored)'
    return f'n={finite_n}'


def _expanded_xlim(*arrays):
    vals = [np.asarray(a, dtype=float) for a in arrays if len(a)]
    if not vals:
        return 0.0, 1.0, 1.08
    all_vals = np.concatenate(vals)
    lo = float(np.nanmin(all_vals))
    hi = float(np.nanmax(all_vals))
    span = hi - lo if hi > lo else 1.0
    return lo, hi, hi + CENSORED_X_PAD * span


def _mark_censored(ax, x_cens, censored_n, color, label, y_frac=0.92):
    if not SHOW_CENSORED_MFPT or censored_n <= 0:
        return
    ax.axvline(x_cens, color=color, ls=':', lw=1.2, alpha=0.8)
    ax.text(x_cens, y_frac, f'{label}\ncensored={censored_n}', color=color,
            fontsize=8, ha='center', va='top', rotation=90,
            transform=ax.get_xaxis_transform())


def _plot_ecdf_with_censoring(ax, vals, censored_n, total_n, color, label, x_cens):
    if vals.size == 0 or total_n == 0:
        return
    x = np.sort(vals)
    y = np.arange(1, x.size + 1) / total_n
    ax.step(x, y, where='post', color=color, lw=2, label=label)
    if SHOW_CENSORED_MFPT and censored_n > 0:
        ax.scatter([x_cens], [x.size / total_n], color=color, marker='>', s=45, zorder=4)
        ax.text(x_cens, x.size / total_n, f' {censored_n}', color=color,
                fontsize=8, ha='left', va='center')


def _diag_per_dataset(df, slice_):
    """Print how many rows survive the slice for each dataset."""
    print(f'slice: {slice_}')
    for d in sorted(df['dataset'].unique()):
        sub = filter_df(df[df['dataset'] == d], **slice_)
        cens = int(sub['mfpt_censored'].sum()) if 'mfpt_censored' in sub.columns else 0
        finite = len(sub) - cens
        flag = '' if len(sub) > 0 else '   <- filtered out!'
        extra = f' ({finite:,} finite, {cens:,} censored)' if cens else ''
        print(f'  {d:50s} {len(sub):>6,} rows{extra}{flag}')
    total = len(filter_df(df, **slice_))
    if total == 0:
        print('\nNo rows matched this slice. Available values for sliced columns:')
        for k in slice_:
            if k in df.columns:
                vals = sorted(df[k].dropna().unique().tolist())
                print(f'  {k}: {vals}')



def plot_ret_vs_ctrls(
    df, slice_=None,
    ret_datasets=RET_DATASETS, ctrl_datasets=CTRL_DATASETS,
    color_map=GROUP_COLORS, kind='kde', show_censored=SHOW_CENSORED_MFPT,
):
    """For each control present in the slice, overlay the pooled RET distribution."""
    sub = filter_df(df, **(slice_ or {}))
    col = 'log10_mfpt'

    ret_frame = sub[sub['dataset'].isin(ret_datasets)]
    ret_vals, ret_cens, ret_total = _mfpt_arrays(ret_frame, col, show_censored=show_censored)
    if ret_vals.size < 5:
        print(f'not enough finite pooled-RET rows in slice (finite n={ret_vals.size}, censored n={ret_cens})')
        return None

    ctrls_present = [c for c in ctrl_datasets if (sub['dataset'] == c).any()]
    if not ctrls_present:
        print('no control datasets in slice')
        return None

    n = len(ctrls_present)
    fig, axes = plt.subplots(2, n, figsize=(4.8 * n, 8.5), squeeze=False)
    for j, ctrl in enumerate(ctrls_present):
        ctrl_frame = sub[sub['dataset'] == ctrl]
        ctrl_vals, ctrl_cens, ctrl_total = _mfpt_arrays(ctrl_frame, col, show_censored=show_censored)
        if ctrl_vals.size < 5:
            axes[0, j].text(0.5, 0.5, f'not enough finite rows\nCTRL finite={ctrl_vals.size}\nCTRL censored={ctrl_cens}',
                            ha='center', va='center', transform=axes[0, j].transAxes)
            axes[0, j].set_axis_off()
            axes[1, j].set_axis_off()
            continue
        ctrl_color = color_map.get(ctrl, CTRL_COLOR)
        ks = ks_2samp(ctrl_vals, ret_vals, alternative='two-sided', mode='auto')
        sep = float(np.median(ret_vals) - np.median(ctrl_vals))
        lo, hi, x_cens = _expanded_xlim(ret_vals, ctrl_vals)

        ax_k = axes[0, j]
        if kind == 'kde':
            sns.kdeplot(ret_vals, ax=ax_k, color=RET_COLOR, lw=2, fill=True, alpha=0.25,
                        label=f'RET pooled ({_censored_label(ret_vals.size, ret_cens, show_censored)})')
            sns.kdeplot(ctrl_vals, ax=ax_k, color=ctrl_color, lw=2, fill=True, alpha=0.25,
                        label=f'{ctrl} ({_censored_label(ctrl_vals.size, ctrl_cens, show_censored)})')
        else:
            bins = np.linspace(lo, hi, 40)
            ax_k.hist(ret_vals, bins=bins, density=True, alpha=0.45, color=RET_COLOR,
                      label=f'RET pooled ({_censored_label(ret_vals.size, ret_cens, show_censored)})')
            ax_k.hist(ctrl_vals, bins=bins, density=True, alpha=0.45, color=ctrl_color,
                      label=f'{ctrl} ({_censored_label(ctrl_vals.size, ctrl_cens, show_censored)})')
        ax_k.axvline(np.median(ret_vals), color=RET_COLOR, ls='--', lw=1.3)
        ax_k.axvline(np.median(ctrl_vals), color=ctrl_color, ls='--', lw=1.3)
        ax_k.set_xlim(lo, x_cens + (x_cens - hi))
        _mark_censored(ax_k, x_cens, ret_cens, RET_COLOR, 'RET', y_frac=0.92)
        _mark_censored(ax_k, x_cens, ctrl_cens, ctrl_color, 'CTRL', y_frac=0.66)
        ax_k.set_xlabel('log10(MFPT)')
        ax_k.set_ylabel('density')
        title_extra = f'\ncensored RET={ret_cens}, CTRL={ctrl_cens}' if show_censored and (ret_cens or ctrl_cens) else ''
        ax_k.set_title(
            f'{ctrl}\nKS={ks.statistic:.3f}, p={ks.pvalue:.2e}\n'
            f'Delta median(log10) = {sep:+.2f}{title_extra}',
            fontsize=10,
        )
        ax_k.legend(fontsize=8, loc='best')

        ax_e = axes[1, j]
        _plot_ecdf_with_censoring(ax_e, ret_vals, ret_cens, ret_total, RET_COLOR,
                                  'RET pooled', x_cens)
        _plot_ecdf_with_censoring(ax_e, ctrl_vals, ctrl_cens, ctrl_total, ctrl_color,
                                  ctrl, x_cens)
        ax_e.set_xlim(lo, x_cens + (x_cens - hi))
        ax_e.set_xlabel('log10(MFPT)')
        ax_e.set_ylabel('ECDF')
        ax_e.set_ylim(0, 1.02)
        ax_e.grid(True, alpha=0.3)
        ax_e.legend(fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.show()
    return fig


# Real sweep axes (free to filter).  Excludes dataset-class-determined ones.
print(f'cross-dataset axes:      {cross_dataset_axes(df)}')
print(f'dataset-determined axes: {dataset_determined_columns(df)}  (do NOT slice on these)')
print(f'SHOW_CENSORED_MFPT:      {SHOW_CENSORED_MFPT}')
print()

# Edit this base slice. Use only cross_dataset_axes.
slice_ = dict(
    prot_p_conc=100.0,
    prot_cooperativity=4.5,
    tnp2_enabled=True,
    tnp2_mu_t0=-8.0,
    tnp2_eps_cpg=1.0,

)

_diag_per_dataset(df, slice_)
df_slice = filter_df(df, **slice_)
print(f'\ntotal base slice rows: {len(df_slice):,}')

if not df_slice.empty:
    fig_ret_ctrl = plot_ret_vs_ctrls(df_slice, show_censored=SHOW_CENSORED_MFPT)


## 4. Sweep along a single axis

Pick one parameter to vary (e.g. `tnp2_mu_t0`, `tnp2_eps_cpg`, `prot_p_conc`, `eads_delta`) and see how the MFPT distribution shifts for each dataset.


In [ ]:
def plot_sweep_axis(df, x, datasets=None, color_map=GROUP_COLORS, kind='violin'):
    """Faceted plot: x-axis = a sweep parameter, one panel per dataset."""
    sub = df if datasets is None else df[df['dataset'].isin(datasets)]
    if x not in sub.columns:
        raise KeyError(f'{x!r} not in DataFrame columns')
    order_ds = sorted(sub['dataset'].unique())
    n = len(order_ds)
    fig, axes = plt.subplots(1, n, figsize=(4 * n + 2, 4.5), sharey=True, squeeze=False)
    for ax, ds in zip(axes[0], order_ds):
        d = sub[sub['dataset'] == ds]
        if d.empty:
            ax.set_visible(False); continue
        x_order = sorted(d[x].dropna().unique())
        if kind == 'violin':
            sns.violinplot(data=d, x=x, y='log10_mfpt', order=x_order, ax=ax,
                           color=color_map.get(ds, 'gray'), cut=0)
        elif kind == 'box':
            sns.boxplot(data=d, x=x, y='log10_mfpt', order=x_order, ax=ax,
                        color=color_map.get(ds, 'gray'), showfliers=False)
        elif kind == 'point':
            grp = d.groupby(x)['log10_mfpt'].agg(['median', 'mean', 'std', 'count']).reset_index()
            ax.errorbar(grp[x], grp['median'], yerr=grp['std'], marker='o',
                        color=color_map.get(ds, 'gray'), capsize=3, lw=1.5)
        ax.set_title(ds.replace('_', '\n'), fontsize=10)
        ax.set_xlabel(x)
    axes[0, 0].set_ylabel('log10(MFPT)')
    plt.tight_layout(); plt.show()


# Auto-pick a varying axis from the *real* (cross-dataset) sweep axes.
candidates = cross_dataset_axes(df)
X = candidates[0] if candidates else None
print(f'cross-dataset axes: {candidates}')
print(f'plotting along: {X}')

if X is not None:
    fixed = {c: df[c].mode().iloc[0] for c in candidates if c != X}
    df_panel = filter_df(df, **fixed)
    print(f'fixed axes: {fixed}')
    print(f'panel rows: {len(df_panel):,}')
    if not df_panel.empty:
        plot_sweep_axis(df_panel, x=X, kind='violin')

## 5. 2D heatmap: median log10(MFPT) over a (×) grid

When two axes vary together (e.g. a (ε_CpG, ΔE_prot) scan), look at how median log10(MFPT) moves across the grid — one heatmap per dataset.

In [ ]:
def plot_heatmap_2d(df, x, y, datasets=None, agg='median'):
    sub = df if datasets is None else df[df['dataset'].isin(datasets)]
    order_ds = sorted(sub['dataset'].unique())
    n = len(order_ds)
    fig, axes = plt.subplots(1, n, figsize=(4 * n + 2, 4), squeeze=False)
    for ax, ds in zip(axes[0], order_ds):
        d = sub[sub['dataset'] == ds]
        if d.empty:
            ax.set_visible(False); continue
        pivot = d.pivot_table(values='log10_mfpt', index=y, columns=x, aggfunc=agg)
        sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f', cmap='viridis', cbar=False)
        ax.set_title(ds.replace('_', '\n'), fontsize=10)
    plt.tight_layout(); plt.show()


# Pick X, Y from the cross-dataset axes (avoids dataset-class-determined ones).
varying = cross_dataset_axes(df)
if len(varying) >= 2:
    X, Y = varying[0], varying[1]
    fixed = {c: df[c].mode().iloc[0] for c in varying if c not in (X, Y)}
    df_grid = filter_df(df, **fixed)
    print(f'heatmap axes: {Y} (rows) × {X} (cols), fixed {fixed}')
    if not df_grid.empty:
        plot_heatmap_2d(df_grid, x=X, y=Y)
else:
    print(f'need >= 2 cross-dataset axes for a heatmap, found {varying}')

## 6. RET vs control comparison: KS test + log10 mean separation

For each control dataset, compute the two-sample KS test and `log10(mean MFPT_RET) - log10(mean MFPT_ctrl)` against pooled RET, scanned across the user-chosen sweep axis.

In [ ]:
def ret_vs_ctrl_table(df, group_keys, ret_datasets=RET_DATASETS, ctrl_datasets=CTRL_DATASETS):
    """For each unique combination of ``group_keys``, KS-test pooled RET vs each control."""
    if not group_keys:
        group_keys = ['_const_']
        df = df.assign(_const_=0)
    rows = []
    for keys, sub in df.groupby(group_keys, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        ret = sub[sub['dataset'].isin(ret_datasets)]['mfpt'].values
        if ret.size < 5:
            continue
        for ctrl in ctrl_datasets:
            c = sub[sub['dataset'] == ctrl]['mfpt'].values
            if c.size < 5:
                continue
            ks = ks_2samp(c, ret, alternative='two-sided', mode='auto')
            rows.append({
                **dict(zip(group_keys, keys)),
                'control': ctrl,
                'n_ctrl': c.size, 'n_ret': ret.size,
                'log10_mean_sep': np.log10(ret.mean()) - np.log10(c.mean()),
                'ks_stat': ks.statistic, 'ks_p': ks.pvalue,
            })
    return pd.DataFrame(rows).drop(columns=['_const_'], errors='ignore')


# Group by the real (cross-dataset) sweep axes - dataset-determined columns
# would confound the comparison.
varying = cross_dataset_axes(df)
print(f'grouping by: {varying or ["(no varying axes - single condition)"]}')
table = ret_vs_ctrl_table(df, group_keys=varying)
display(table.head(20))

In [ ]:
def plot_separation_curve(table, x, hue='control'):
    if table.empty or x not in table.columns:
        print(f'nothing to plot for x={x}'); return
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(data=table, x=x, y='log10_mean_sep', hue=hue, marker='o', ax=a1)
    a1.axhline(0, color='k', lw=0.7, ls='--')
    a1.set_ylabel('log10(mean RET) - log10(mean ctrl)')
    a1.legend(fontsize=8)

    sns.lineplot(data=table, x=x, y='ks_p', hue=hue, marker='o', ax=a2)
    a2.set_yscale('log')
    a2.set_ylabel('KS p-value (log scale)')
    a2.legend(fontsize=8)
    plt.tight_layout(); plt.show()

if varying and not table.empty:
    plot_separation_curve(table, x=varying[0])

## 7. CpG-binned MFPT (composition-aware)

Requires section 2 (sequence features attached).  At a fixed parameter slice, plot median log10(MFPT) per CpG-density bin for each dataset — the central diagnostic for the v=0 / v=1 swap test.

In [ ]:
def plot_mfpt_vs_cpg(df, datasets=None, color_map=GROUP_COLORS, n_bins=10, by='rho_cpg'):
    sub = df if datasets is None else df[df['dataset'].isin(datasets)]
    sub = sub.dropna(subset=[by, 'log10_mfpt'])
    if sub.empty:
        print('no rows with sequence features'); return
    bins = np.linspace(sub[by].min(), sub[by].max(), n_bins + 1)
    mids = 0.5 * (bins[:-1] + bins[1:])
    fig, ax = plt.subplots(figsize=(9, 5))
    for ds in sorted(sub['dataset'].unique()):
        d = sub[sub['dataset'] == ds].copy()
        d['bin'] = pd.cut(d[by], bins=bins, labels=mids, include_lowest=True)
        agg = d.groupby('bin', observed=True)['log10_mfpt'].median()
        ax.plot(agg.index.astype(float), agg.values, marker='o', lw=1.5,
                color=color_map.get(ds, 'gray'), label=ds)
    ax.set_xlabel(by); ax.set_ylabel('median log10(MFPT)')
    ax.legend(fontsize=8, loc='best'); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

if 'rho_cpg' in df.columns and df['rho_cpg'].notna().any():
    df_cpg = filter_df(df, **slice_)
    plot_mfpt_vs_cpg(df_cpg)
else:
    print('no sequence features - run section 2 first')

## 8. Quick numerical summary

Convenience pivot for the slice you're inspecting.

In [ ]:
if 'df_slice' in globals() and not df_slice.empty:
    summary = (df_slice.groupby('dataset')['log10_mfpt']
                       .agg(['count', 'median', 'mean', 'std',
                             lambda v: np.percentile(v, 25),
                             lambda v: np.percentile(v, 75)])
                       .rename(columns={'<lambda_0>': 'p25', '<lambda_1>': 'p75'})
                       .round(3))
    display(summary)


## 9. CpG-dependent TNP2 cooperativity disruption

The J_eff model does not compare symmetric/asymmetric rate scaling. TNP2 now acts through CpG-dependent reduction of protamine cooperativity, so this section checks MFPT trends against sequence CpG features and TNP2 sweep parameters.


In [ ]:
def plot_tnp2_cpg_diagnostic(df, slice_=None, color_map=GROUP_COLORS):
    """Plot log10(MFPT) against CpG features for enabled TNP2 J_eff runs."""
    sub = filter_df(df, **(slice_ or {}))
    if 'tnp2_enabled' in sub.columns:
        sub = sub[sub['tnp2_enabled'] == True]
    required = {'log10_mfpt', 'n_cpg', 'dataset'}
    missing = sorted(required - set(sub.columns))
    if missing:
        print(f'missing columns: {missing}; attach sequence features first')
        return None
    sub = sub[np.isfinite(sub['log10_mfpt']) & sub['n_cpg'].notna()].copy()
    if sub.empty:
        print('no finite TNP2 rows with CpG features in this slice')
        return None

    x_cols = ['n_cpg']
    if 'rho_cpg' in sub.columns and sub['rho_cpg'].notna().any():
        x_cols.append('rho_cpg')
    sweep_cols = [c for c in ('tnp2_mu_t0', 'tnp2_eps_cpg') if c in sub.columns and sub[c].nunique(dropna=False) > 1]

    n_panels = len(x_cols) + len(sweep_cols)
    fig, axes = plt.subplots(1, n_panels, figsize=(5.2 * n_panels, 4.5), squeeze=False)
    axes = axes.ravel()

    panel = 0
    for x_col in x_cols:
        ax = axes[panel]
        for ds in sorted(sub['dataset'].unique()):
            d = sub[sub['dataset'] == ds]
            ax.scatter(d[x_col], d['log10_mfpt'], s=8, alpha=0.45,
                       color=color_map.get(ds, 'gray'), label=ds, rasterized=True)
            if d[x_col].nunique() >= 4:
                bins = np.linspace(d[x_col].min(), d[x_col].max(), 11)
                mids = 0.5 * (bins[:-1] + bins[1:])
                d2 = d.copy()
                d2['bin'] = pd.cut(d2[x_col], bins=bins, labels=mids, include_lowest=True)
                agg = d2.groupby('bin', observed=True)['log10_mfpt'].median()
                ax.plot(agg.index.astype(float), agg.values, marker='o', lw=2,
                        color=color_map.get(ds, 'gray'))
        ax.set_xlabel(x_col)
        ax.set_ylabel('log10(MFPT)')
        ax.set_title(f'MFPT vs {x_col}')
        ax.grid(True, alpha=0.3)
        panel += 1

    for x_col in sweep_cols:
        ax = axes[panel]
        sns.boxplot(data=sub, x=x_col, y='log10_mfpt', hue='dataset', ax=ax,
                    palette={k: color_map.get(k, 'gray') for k in sub['dataset'].unique()})
        ax.set_title(f'MFPT by {x_col}')
        ax.grid(True, axis='y', alpha=0.3)
        panel += 1

    axes[0].legend(fontsize=8, loc='best')
    plt.tight_layout()
    plt.show()
    return fig


slice_tnp2_cpg = {k: v for k, v in slice_.items() if k in df.columns}
fig_tnp2_cpg = plot_tnp2_cpg_diagnostic(df, slice_tnp2_cpg)
